In [ ]:
ENSEMBLE_MEMBERS = 600
MODEL = "WITCH 5.0"
SCENARIO = "COMMIT-2°C-2030_NZCO2"
FILE = "101_netzero_co2_extended.csv"
MAGICC_FLAG = "GHG"
N_PROCESSES = 2

# 303: Run MAGICC (missing/failed reruns)
In this notebook, we will run MAGICC for a single scenario for a small number of ensemble members for MAGICC7.

In [18]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("pint.util").setLevel(logging.ERROR)

import multiprocessing
import os
import platform
from functools import partial
from pathlib import Path

import gcages.cmip7_scenariomip.pre_processing.reaggregation.basic
from gcages.cmip7_scenariomip.harmonisation import (
    create_cmip7_scenariomip_global_harmoniser,
)
from gcages.cmip7_scenariomip.infilling import CMIP7ScenarioMIPInfiller
from gcages.cmip7_scenariomip.post_processing import CMIP7ScenarioMIPPostProcessor
from gcages.cmip7_scenariomip.pre_processing import CMIP7ScenarioMIPPreProcessor
from gcages.cmip7_scenariomip.scm_running import CMIP7ScenarioMIPSCMRunner
from gcages.index_manipulation import split_sectors

import dotenv
import openscm_units
import pandas_indexing as pix
import pandas_openscm
import pint
import seaborn as sns
from pandas_openscm.io import load_timeseries_csv
import pandas as pd

# Papermill output handling
import sys
from datetime import datetime

# Import sanitization function
from utils import sanitize_label

# Load environment variables
dotenv.load_dotenv()

# Set up output folder
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

In [19]:
BASE_DIR = Path("../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs")
CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS = BASE_DIR / "history_cmip7_scenariomip.csv"

In [20]:
FILE = OUTPUT_FOLDER / FILE

In [21]:
start = load_timeseries_csv(
    FILE,
    index_columns=["model", "scenario", "region", "variable", "unit"],
    out_columns_type=int,
    out_columns_name="year"
)

In [22]:
scenario_to_run = (
    start
    .loc[
        pd.IndexSlice[MODEL, SCENARIO, :, :, :], :
    ]
)

In [23]:
scenario_to_run

year                                                                                    2023  \
model     scenario              region variable                unit                            
WITCH 5.0 COMMIT-2°C-2030_NZCO2 World  Emissions|BC            Mt BC/yr             7.219245   
                                       Emissions|C2F6          kt C2F6/yr           2.431873   
                                       Emissions|C3F8          kt C3F8/yr           0.721178   
                                       Emissions|C4F10         kt C4F10/yr          0.101586   
                                       Emissions|C5F12         kt C5F12/yr          0.061383   
                                       Emissions|C6F14         kt C6F14/yr          0.062998   
                                       Emissions|C7F16         kt C7F16/yr          0.186453   
                                       Emissions|C8F18         kt C8F18/yr          0.136638   
                                       Emissions|CCl4          kt CCl4/yr          40.297308   
                                       Emissions|CF4           kt CF4/yr           15.117718   
                                       Emissions|CFC11         kt CFC11/yr         55.993337   
                                       Emissions|CFC113        kt CFC113/yr         2.030656   
                                       Emissions|CFC114        kt CFC114/yr         0.873953   
                                       Emissions|CFC115        kt CFC115/yr         0.027629   
                                       Emissions|CFC12         kt CFC12/yr         20.278901   
                                       Emissions|CH2Cl2        kt CH2Cl2/yr      1612.125558   
                                       Emissions|CH3Br         kt CH3Br/yr        140.486974   
                                       Emissions|CH3CCl3       kt CH3CCl3/yr        1.925368   
                                       Emissions|CH3Cl         kt CH3Cl/yr       5376.602241   
                                       Emissions|CH4           Mt CH4/yr          363.489901   
                                       Emissions|CHCl3         kt CHCl3/yr        400.499380   
                                       Emissions|CO            Mt CO/yr           774.053434   
                                       Emissions|CO2|Biosphere Mt CO2/yr         3627.550667   
                                       Emissions|CO2|Fossil    Mt CO2/yr        38712.217040   
                                       Emissions|HCFC141b      kt HCFC141b/yr      57.429166   
                                       Emissions|HCFC142b      kt HCFC142b/yr      20.176795   
                                       Emissions|HCFC22        kt HCFC22/yr       324.526866   
                                       Emissions|HFC125        kt HFC125/yr       124.074300   
                                       Emissions|HFC134a       kt HFC134a/yr      291.788600   
                                       Emissions|HFC143a       kt HFC143a/yr       35.833800   
                                       Emissions|HFC152a       kt HFC152a/yr       54.869900   
                                       Emissions|HFC227ea      kt HFC227ea/yr       6.645300   
                                       Emissions|HFC23         kt HFC23/yr         14.051843   
                                       Emissions|HFC236fa      kt HFC236fa/yr       0.354300   
                                       Emissions|HFC245fa      kt HFC245fa/yr      15.504500   
                                       Emissions|HFC32         kt HFC32/yr        105.566000   
                                       Emissions|HFC365mfc     kt HFC365mfc/yr      5.241900   
                                       Emissions|HFC4310mee    kt HFC4310/yr        0.882600   
                                       Emissions|Halon1202     kt Halon1202/yr      0.000000   
                                       Emissions|Halon1211     k

In [24]:
MAGICC_EXE_PATH = Path(
    "../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs/magicc-v7.6.0a3/bin"
)
MAGICC_CMIP7_SCENARIOMIP_PROBABILISTIC_CONFIG_FILE = Path(
    "../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs/magicc-v7.6.0a3/configs/magicc-ar7-fast-track-drawnset-v0-3-0.json"
)

In [25]:
if platform.system() == "Darwin":
    if platform.processor() == "arm":
        MAGICC_EXE = MAGICC_EXE_PATH / "magicc-darwin-arm64"
        os.environ["DYLD_LIBRARY_PATH"] = "/opt/homebrew/Cellar/gcc/15.2.0_1/lib/gcc/current"

elif platform.system() == "Linux":
    MAGICC_EXE = MAGICC_EXE_PATH / "magicc"

elif platform.system() == "Windows":
    MAGICC_EXE = MAGICC_EXE_PATH / "magicc.exe"

In [26]:
scm_runner = CMIP7ScenarioMIPSCMRunner.from_cmip7_scenariomip_config(
    n_processes=N_PROCESSES,
    magicc_exe_path=MAGICC_EXE,
    magicc_prob_distribution_path=MAGICC_CMIP7_SCENARIOMIP_PROBABILISTIC_CONFIG_FILE,
    historical_emissions_path=CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS,
    output_variables=("Surface Air Temperature Change",),
    batch_size_scenarios=15,
)

In [27]:
scm_runner.climate_models_cfgs["MAGICC7"] = scm_runner.climate_models_cfgs[
        "MAGICC7"
][:ENSEMBLE_MEMBERS]

In [28]:
magicc7_cfgs = [
    {**c, "rf_total_runmodus": MAGICC_FLAG}
    for c in scm_runner.climate_models_cfgs["MAGICC7"]
]

In [29]:
scm_runner.climate_models_cfgs["MAGICC7"] = magicc7_cfgs

In [30]:
scm_results = scm_runner(scenario_to_run)
scm_results

Scenario batch:   0%|                                     | 0/1 [00:00<?, ?it/s]

Climate models:   0%|                               | 0.00/1.00 [00:00<?, ?it/s][WARNING] 16:55:02 - openscm_runner.adapters.magicc7.magicc7: Historical data has not been checked



Writing SCEN7 files: 100%|███████████████████| 1.00/1.00 [00:00<00:00, 5.85it/s]



Front serial:   0%|                                 | 0.00/2.00 [00:00<?, ?it/s]/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
Front serial: 100%|██████████████████████████| 2.00/2.00 [00:03<00:00, 1.69s/it]



Front parallel:   0%|                               | 0.00/2.00 [00:00<?, ?it/s]/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: Th

time                                                                                              1750  \
climate_model  model     region run_id scenario              unit variable                               
MAGICCv7.6.0a3 WITCH 5.0 World  0      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change   0.0   
                                1      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change   0.0   
                                2      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change   0.0   
                                3      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change   0.0   
                                4      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change   0.0   
...                                                                                                ...   
                                595    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change   0.0   
                                596    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change   0.0   
                                597    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change   0.0   
                                598    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change   0.0   
                                599    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change   0.0   

time                                                                                                  1751  \
climate_model  model     region run_id scenario              unit variable                                   
MAGICCv7.6.0a3 WITCH 5.0 World  0      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000156   
                                1      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000126   
                                2      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000123   
                                3      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000140   
                                4      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000135   
...                                                                                                    ...   
                                595    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000120   
                                596    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000154   
                                597    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000122   
                                598    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000124   
                                599    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000113   

time                                                                                                  1752  \
climate_model  model     region run_id scenario              unit variable                                   
MAGICCv7.6.0a3 WITCH 5.0 World  0      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000495   
                                1      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000400   
                                2      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000388   
                                3      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000443   
                                4      COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000431   
...                                                                                                    ...   
                                595    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000363   
                                596    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature Change  0.000516   
                                597    COMMIT-2°C-2030_NZCO2 K    Surface Air Temperature 

In [31]:
post_processor = CMIP7ScenarioMIPPostProcessor.from_cmip7_scenariomip_config()
post_processed_results = post_processor(
    scm_results.loc[pix.isin(variable=["Surface Air Temperature Change"])]
)

In [32]:
post_processed_results.timeseries_run_id

time                                                                                              1750  \
climate_model  model     region run_id scenario              unit variable                               
MAGICCv7.6.0a3 WITCH 5.0 World  0      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.596485   
                                1      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.660739   
                                2      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.592888   
                                3      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.567012   
                                4      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.598755   
...                                                                                                ...   
                                595    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.557948   
                                596    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.668519   
                                597    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.602431   
                                598    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.552962   
                                599    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.571630   

time                                                                                              1751  \
climate_model  model     region run_id scenario              unit variable                               
MAGICCv7.6.0a3 WITCH 5.0 World  0      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.596329   
                                1      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.660613   
                                2      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.592765   
                                3      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.566872   
                                4      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.598619   
...                                                                                                ...   
                                595    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.557828   
                                596    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.668365   
                                597    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.602309   
                                598    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.552838   
                                599    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.571517   

time                                                                                              1752  \
climate_model  model     region run_id scenario              unit variable                               
MAGICCv7.6.0a3 WITCH 5.0 World  0      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.595990   
                                1      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.660339   
                                2      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.592500   
                                3      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.566570   
                                4      COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.598323   
...                                                                                                ...   
                                595    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.557585   
                                596    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.668003   
                                597    COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT) -0.602045   
                                598    COMMIT-2°C-2030_NZCO2 K    Surface Te

In [33]:
post_processed_results.metadata_quantile.loc[
    pix.isin(quantile=[0.05, 0.5, 0.95])
].unstack(["quantile", "metric"]).round(2).sort_index(axis="columns")

quantile                                                                               0.05  \
metric                                                                                 2100   
climate_model  model     region scenario              unit variable                           
MAGICCv7.6.0a3 WITCH 5.0 World  COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT)  0.77   
                                                      yr   Surface Temperature (GSAT)   NaN   

quantile                                                                                     \
metric                                                                                  max   
climate_model  model     region scenario              unit variable                           
MAGICCv7.6.0a3 WITCH 5.0 World  COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT)  0.95   
                                                      yr   Surface Temperature (GSAT)   NaN   

quantile                                                                                        \
metric                                                                                max_year   
climate_model  model     region scenario              unit variable                              
MAGICCv7.6.0a3 WITCH 5.0 World  COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT)      NaN   
                                                      yr   Surface Temperature (GSAT)   2037.0   

quantile                                                                               0.50  \
metric                                                                                 2100   
climate_model  model     region scenario              unit variable                           
MAGICCv7.6.0a3 WITCH 5.0 World  COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT)  1.37   
                                                      yr   Surface Temperature (GSAT)   NaN   

quantile                                                                                     \
metric                                                                                  max   
climate_model  model     region scenario              unit variable                           
MAGICCv7.6.0a3 WITCH 5.0 World  COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT)  1.56   
                                                      yr   Surface Temperature (GSAT)   NaN   

quantile                                                                                        \
metric                                                                                max_year   
climate_model  model     region scenario              unit variable                              
MAGICCv7.6.0a3 WITCH 5.0 World  COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT)      NaN   
                                                      yr   Surface Temperature (GSAT)   2043.0   

quantile                                                                               0.95  \
metric                                                                                 2100   
climate_model  model     region scenario              unit variable                           
MAGICCv7.6.0a3 WITCH 5.0 World  COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT)  2.45   
                                                      yr   Surface Temperature (GSAT)   NaN   

quantile                                                                                    \
metric                                                                                 max   
climate_model  model     region scenario              unit variable                          
MAGICCv7.6.0a3 WITCH 5.0 World  COMMIT-2°C-2030_NZCO2 K    Surface Temperature (GSAT)  2.5   
                                                      yr   Surface Temperature (GSAT)  NaN   

quantile                                                                                        
metric                                                                

In [ ]:
# Create output directory
output_dir = OUTPUT_FOLDER / "individual_runs_missing"
output_dir.mkdir(parents=True, exist_ok=True)

# Sanitize labels
sanitized_model = sanitize_label(MODEL)
sanitized_scenario = sanitize_label(SCENARIO)

# Generate output filename
output_filename = f"{sanitized_model}_{sanitized_scenario}_{MAGICC_FLAG}.csv"
output_path = output_dir / output_filename

post_processed_results.timeseries_run_id.to_csv(output_path)

# Print run details (manifest is now handled by 302_running_helpers.py per session)
print(f"Model: {MODEL}")
print(f"Scenario: {SCENARIO}")
print(f"MAGICC flag: {MAGICC_FLAG}")
print(f"Sanitized model: {sanitized_model}")
print(f"Sanitized scenario: {sanitized_scenario}")
print(f"Output saved: {output_path}")